In [1]:
import math
import random
import itertools
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms

# ============================================================
# Device
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

# ============================================================
# Global settings
# ============================================================
NUM_SEEDS = 5
SEEDS = [42, 43, 44, 45, 46]

hidden_dim = 16
epochs = 30
lr = 1e-3

batch_size_train = 128
batch_size_eval = 256
target_acc = 0.85
train_subset_size = 10000

# 95% CI t-critical for df=5 (n=4)
T_CRIT_95_DF4 = 2.776

# ============================================================
# Grid search ranges
# ============================================================
lambda_grid = [0.0, 0.01, 0.02, 0.05]
prototype_grid = [4, 8, 12]
tau_grid = [0.5, 1.0, 3.0]

# If you want a smaller quick grid, uncomment this instead:
# lambda_grid = [0.0, 0.02, 0.05]
# prototype_grid = [4, 8]
# tau_grid = [1.0, 3.0]

# ============================================================
# Reproducibility helper
# ============================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

# Fashion-MNIST loaders
# ============================================================
def make_fashion_mnist_loaders(seed: int):
    set_seed(seed)

    transform = transforms.Compose([
        transforms.ToTensor()
    ])

    train_full = datasets.FashionMNIST(
        root="./data",
        train=True,
        download=True,
        transform=transform
    )

    test_set = datasets.FashionMNIST(
        root="./data",
        train=False,
        download=True,
        transform=transform
    )

    if train_subset_size is not None and train_subset_size < len(train_full):
        g_subset = torch.Generator().manual_seed(seed)
        indices = torch.randperm(len(train_full), generator=g_subset)[:train_subset_size]
        train_full = Subset(train_full, indices.tolist())

    n_total = len(train_full)
    n_val = int(0.1 * n_total)
    n_train = n_total - n_val

    g_split = torch.Generator().manual_seed(seed)
    train_set, val_set = random_split(train_full, [n_train, n_val], generator=g_split)

    g_loader = torch.Generator().manual_seed(seed)

    train_loader = DataLoader(
        train_set,
        batch_size=batch_size_train,
        shuffle=True,
        generator=g_loader,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    val_loader = DataLoader(
        val_set,
        batch_size=batch_size_eval,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    test_loader = DataLoader(
        test_set,
        batch_size=batch_size_eval,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    return train_loader, val_loader, test_loader

# ============================================================
# Models
# ============================================================
class StandardMLP(nn.Module):
    def __init__(self, in_dim=28*28, hidden_dim=16, out_dim=10):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        h = torch.tanh(self.fc1(x))
        logits = self.fc2(h)
        return logits, h


class GeometryLocalNet(nn.Module):
    def __init__(self, in_dim=28*28, hidden_dim=16, out_dim=10, num_prototypes=8, tau=1.0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.out_dim = out_dim
        self.K = num_prototypes
        self.tau = tau

        self.fc1 = nn.Linear(in_dim, hidden_dim)

        self.prototypes = nn.Parameter(torch.randn(num_prototypes, hidden_dim) * 0.5)
        self.phi = nn.Parameter(torch.randn(num_prototypes, hidden_dim) * 0.5)

        self.expert_W = nn.Parameter(torch.randn(num_prototypes, out_dim, hidden_dim) * 0.15)
        self.expert_b = nn.Parameter(torch.zeros(num_prototypes, out_dim))

    def memberships(self, h):
        dist_sq = torch.cdist(h, self.prototypes, p=2.0) ** 2
        g = F.softmax(-dist_sq / self.tau, dim=1)
        return g

    def forward(self, x):
        x = x.view(x.size(0), -1)
        h = torch.tanh(self.fc1(x))
        g = self.memberships(h)

        expert_logits = torch.einsum("kch,bh->bkc", self.expert_W, h) + self.expert_b.unsqueeze(0)
        logits = torch.sum(g.unsqueeze(-1) * expert_logits, dim=1)

        local_target = g @ self.phi
        local_loss = 0.5 * torch.mean(torch.sum((h - local_target) ** 2, dim=1))

        return logits, h, g, local_loss

# ============================================================
# Utilities
# ============================================================
def evaluate_standard(model, loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_n = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            logits, _ = model(xb)
            loss = F.cross_entropy(logits, yb)

            total_loss += loss.item() * xb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_n += xb.size(0)

    return total_loss / total_n, total_correct / total_n


def evaluate_geometry(model, loader, lambda_local):
    model.eval()
    total_loss = 0.0
    total_ce = 0.0
    total_loc = 0.0
    total_correct = 0
    total_n = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            logits, _, _, local_loss = model(xb)
            ce = F.cross_entropy(logits, yb)
            loss = ce + lambda_local * local_loss

            total_loss += loss.item() * xb.size(0)
            total_ce += ce.item() * xb.size(0)
            total_loc += local_loss.item() * xb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_n += xb.size(0)

    return {
        "loss": total_loss / total_n,
        "ce": total_ce / total_n,
        "local": total_loc / total_n,
        "acc": total_correct / total_n
    }


def grad_norm(module):
    sq = 0.0
    for p in module.parameters():
        if p.grad is not None:
            sq += p.grad.detach().pow(2).sum().item()
    return math.sqrt(sq + 1e-12)


def epochs_to_threshold(acc_curve, threshold=0.85):
    for i, a in enumerate(acc_curve, start=1):
        if a >= threshold:
            return i
    return np.nan


def mean_ci95(arr):
    arr = np.asarray(arr, dtype=np.float64)
    mean = np.mean(arr)
    if len(arr) <= 1:
        return mean, 0.0
    std = np.std(arr, ddof=1)
    ci = T_CRIT_95_DF4  * std / np.sqrt(len(arr))
    return mean, ci


def fmt_mean_ci(mean_val, ci_val, digits=4):
    return f"{mean_val:.{digits}f} ± {ci_val:.{digits}f}"

# ============================================================
# Training
# ============================================================
def train_standard(model, train_loader, val_loader, epochs=80, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    hist = {
        "train_loss": [],
        "val_loss": [],
        "val_acc": [],
        "batch_loss_std": [],
        "batch_grad_std": [],
    }

    for epoch in range(epochs):
        model.train()
        batch_losses = []
        batch_grad_norms = []

        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad()
            logits, _ = model(xb)
            loss = F.cross_entropy(logits, yb)
            loss.backward()

            batch_grad_norms.append(grad_norm(model.fc1))
            opt.step()
            batch_losses.append(loss.item())

        train_loss = float(np.mean(batch_losses))
        val_loss, val_acc = evaluate_standard(model, val_loader)

        hist["train_loss"].append(train_loss)
        hist["val_loss"].append(val_loss)
        hist["val_acc"].append(val_acc)
        hist["batch_loss_std"].append(float(np.std(batch_losses)))
        hist["batch_grad_std"].append(float(np.std(batch_grad_norms)))

    return hist


def train_geometry(model, train_loader, val_loader, epochs=80, lr=1e-3, lambda_local=0.02):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    hist = {
        "train_loss": [],
        "val_loss": [],
        "val_acc": [],
        "batch_loss_std": [],
        "batch_grad_std": [],
    }

    for epoch in range(epochs):
        model.train()
        batch_losses = []
        batch_grad_norms = []

        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad()
            logits, _, _, local_loss = model(xb)
            ce = F.cross_entropy(logits, yb)
            loss = ce + lambda_local * local_loss
            loss.backward()

            batch_grad_norms.append(grad_norm(model.fc1))
            opt.step()
            batch_losses.append(loss.item())

        train_loss = float(np.mean(batch_losses))
        val_metrics = evaluate_geometry(model, val_loader, lambda_local=lambda_local)

        hist["train_loss"].append(train_loss)
        hist["val_loss"].append(val_metrics["loss"])
        hist["val_acc"].append(val_metrics["acc"])
        hist["batch_loss_std"].append(float(np.std(batch_losses)))
        hist["batch_grad_std"].append(float(np.std(batch_grad_norms)))

    return hist

# ============================================================
# Baseline: Standard BP over 2 seeds
# ============================================================
def run_standard_bp():
    print("\n================ Running Standard BP baseline ================\n")
    seed_metrics = []

    for seed in SEEDS:
        print(f"Seed {seed}")
        set_seed(seed)
        train_loader, val_loader, test_loader = make_fashion_mnist_loaders(seed)

        model = StandardMLP(in_dim=28*28, hidden_dim=hidden_dim, out_dim=10)
        hist = train_standard(model, train_loader, val_loader, epochs=epochs, lr=lr)

        test_loss, test_acc = evaluate_standard(model, test_loader)
        val_best_acc = float(np.max(hist["val_acc"]))
        final_val_acc = float(hist["val_acc"][-1])
        final_val_loss = float(hist["val_loss"][-1])

        metrics = {
            "seed": seed,
            "val_best_acc": val_best_acc,
            "final_val_acc": final_val_acc,
            "final_val_loss": final_val_loss,
            "test_loss": test_loss,
            "test_acc": test_acc,
            "epochs_to_target": float(epochs_to_threshold(hist["val_acc"], threshold=target_acc)),
            "avg_batch_loss_std": float(np.mean(hist["batch_loss_std"])),
            "avg_hidden_grad_std": float(np.mean(hist["batch_grad_std"])),
        }
        seed_metrics.append(metrics)

        print(
            f"  test_loss={test_loss:.4f} | test_acc={test_acc:.4f} | "
            f"best_val_acc={val_best_acc:.4f} | epochs_to_target={metrics['epochs_to_target']:.0f}"
        )

    return seed_metrics

# ============================================================
# Geometry-local grid search over 2 seeds
# ============================================================
def run_geometry_grid():
    print("\n================ Running Geometry-local grid search ================\n")
    all_cfg_results = []

    grid = list(itertools.product(lambda_grid, prototype_grid, tau_grid))
    print(f"Total geometry-local configurations: {len(grid)}")

    for cfg_idx, (lam, nproto, tau_val) in enumerate(grid, start=1):
        print(
            f"\n[{cfg_idx:02d}/{len(grid)}] "
            f"lambda={lam}, num_prototypes={nproto}, tau={tau_val}"
        )

        seed_metrics = []

        for seed in SEEDS:
            set_seed(seed)
            train_loader, val_loader, test_loader = make_fashion_mnist_loaders(seed)

            model = GeometryLocalNet(
                in_dim=28*28,
                hidden_dim=hidden_dim,
                out_dim=10,
                num_prototypes=nproto,
                tau=tau_val
            )

            hist = train_geometry(
                model,
                train_loader,
                val_loader,
                epochs=epochs,
                lr=lr,
                lambda_local=lam
            )

            test_metrics = evaluate_geometry(model, test_loader, lambda_local=lam)

            val_best_acc = float(np.max(hist["val_acc"]))
            final_val_acc = float(hist["val_acc"][-1])
            final_val_loss = float(hist["val_loss"][-1])

            metrics = {
                "seed": seed,
                "val_best_acc": val_best_acc,
                "final_val_acc": final_val_acc,
                "final_val_loss": final_val_loss,
                "test_loss": test_metrics["loss"],
                "test_acc": test_metrics["acc"],
                "test_ce": test_metrics["ce"],
                "test_local": test_metrics["local"],
                "epochs_to_target": float(epochs_to_threshold(hist["val_acc"], threshold=target_acc)),
                "avg_batch_loss_std": float(np.mean(hist["batch_loss_std"])),
                "avg_hidden_grad_std": float(np.mean(hist["batch_grad_std"])),
            }
            seed_metrics.append(metrics)

        # Aggregate across 2 seeds
        agg = {
            "lambda_local": lam,
            "num_prototypes": nproto,
            "tau": tau_val,
            "val_best_acc_mean": np.mean([m["val_best_acc"] for m in seed_metrics]),
            "val_best_acc_ci": mean_ci95([m["val_best_acc"] for m in seed_metrics])[1],
            "final_val_acc_mean": np.mean([m["final_val_acc"] for m in seed_metrics]),
            "final_val_acc_ci": mean_ci95([m["final_val_acc"] for m in seed_metrics])[1],
            "final_val_loss_mean": np.mean([m["final_val_loss"] for m in seed_metrics]),
            "final_val_loss_ci": mean_ci95([m["final_val_loss"] for m in seed_metrics])[1],
            "test_acc_mean": np.mean([m["test_acc"] for m in seed_metrics]),
            "test_acc_ci": mean_ci95([m["test_acc"] for m in seed_metrics])[1],
            "test_loss_mean": np.mean([m["test_loss"] for m in seed_metrics]),
            "test_loss_ci": mean_ci95([m["test_loss"] for m in seed_metrics])[1],
            "test_ce_mean": np.mean([m["test_ce"] for m in seed_metrics]),
            "test_ce_ci": mean_ci95([m["test_ce"] for m in seed_metrics])[1],
            "test_local_mean": np.mean([m["test_local"] for m in seed_metrics]),
            "test_local_ci": mean_ci95([m["test_local"] for m in seed_metrics])[1],
            "epochs_to_target_mean": np.nanmean([m["epochs_to_target"] for m in seed_metrics]),
            "epochs_to_target_ci": mean_ci95([m["epochs_to_target"] for m in seed_metrics])[1] if not np.any(np.isnan([m["epochs_to_target"] for m in seed_metrics])) else np.nan,
            "avg_batch_loss_std_mean": np.mean([m["avg_batch_loss_std"] for m in seed_metrics]),
            "avg_batch_loss_std_ci": mean_ci95([m["avg_batch_loss_std"] for m in seed_metrics])[1],
            "avg_hidden_grad_std_mean": np.mean([m["avg_hidden_grad_std"] for m in seed_metrics]),
            "avg_hidden_grad_std_ci": mean_ci95([m["avg_hidden_grad_std"] for m in seed_metrics])[1],
            "seed_metrics": seed_metrics,
        }

        all_cfg_results.append(agg)

        print(
            f"  val_best_acc={agg['val_best_acc_mean']:.4f} ± {agg['val_best_acc_ci']:.4f} | "
            f"test_acc={agg['test_acc_mean']:.4f} ± {agg['test_acc_ci']:.4f} | "
            f"test_loss={agg['test_loss_mean']:.4f} ± {agg['test_loss_ci']:.4f}"
        )

    return all_cfg_results

# ============================================================
# Run baseline and grid
# ============================================================
standard_seed_metrics = run_standard_bp()
geometry_results = run_geometry_grid()

# ============================================================
# Summarize baseline
# ============================================================
std_test_loss_mean, std_test_loss_ci = mean_ci95([m["test_loss"] for m in standard_seed_metrics])
std_test_acc_mean, std_test_acc_ci = mean_ci95([m["test_acc"] for m in standard_seed_metrics])
std_val_best_acc_mean, std_val_best_acc_ci = mean_ci95([m["val_best_acc"] for m in standard_seed_metrics])
std_final_val_acc_mean, std_final_val_acc_ci = mean_ci95([m["final_val_acc"] for m in standard_seed_metrics])
std_final_val_loss_mean, std_final_val_loss_ci = mean_ci95([m["final_val_loss"] for m in standard_seed_metrics])
std_batch_loss_mean, std_batch_loss_ci = mean_ci95([m["avg_batch_loss_std"] for m in standard_seed_metrics])
std_hidden_grad_mean, std_hidden_grad_ci = mean_ci95([m["avg_hidden_grad_std"] for m in standard_seed_metrics])

print("\n================ Standard BP summary over 2 seeds ================\n")
print(f"best_val_acc         : {fmt_mean_ci(std_val_best_acc_mean, std_val_best_acc_ci)}")
print(f"final_val_acc        : {fmt_mean_ci(std_final_val_acc_mean, std_final_val_acc_ci)}")
print(f"final_val_loss       : {fmt_mean_ci(std_final_val_loss_mean, std_final_val_loss_ci)}")
print(f"test_acc             : {fmt_mean_ci(std_test_acc_mean, std_test_acc_ci)}")
print(f"test_loss            : {fmt_mean_ci(std_test_loss_mean, std_test_loss_ci)}")
print(f"avg_batch_loss_std   : {fmt_mean_ci(std_batch_loss_mean, std_batch_loss_ci)}")
print(f"avg_hidden_grad_std  : {fmt_mean_ci(std_hidden_grad_mean, std_hidden_grad_ci)}")

# ============================================================
# Rank geometry-local configurations
# Primary sort: mean best val acc (descending)
# Secondary: mean test acc (descending)
# Tertiary: mean test loss (ascending)
# ============================================================
geometry_results_sorted = sorted(
    geometry_results,
    key=lambda d: (-d["val_best_acc_mean"], -d["test_acc_mean"], d["test_loss_mean"])
)

print("\n================ Top 10 geometry-local configurations ================\n")
top_k = min(10, len(geometry_results_sorted))
for rank, r in enumerate(geometry_results_sorted[:top_k], start=1):
    print(
        f"[{rank:02d}] lambda={r['lambda_local']}, protos={r['num_prototypes']}, tau={r['tau']} | "
        f"best_val_acc={fmt_mean_ci(r['val_best_acc_mean'], r['val_best_acc_ci'])} | "
        f"test_acc={fmt_mean_ci(r['test_acc_mean'], r['test_acc_ci'])} | "
        f"test_loss={fmt_mean_ci(r['test_loss_mean'], r['test_loss_ci'])}"
    )

best_cfg = geometry_results_sorted[0]

print("\n================ Best geometry-local configuration ================\n")
print(f"lambda_local         : {best_cfg['lambda_local']}")
print(f"num_prototypes       : {best_cfg['num_prototypes']}")
print(f"tau                  : {best_cfg['tau']}")
print(f"best_val_acc         : {fmt_mean_ci(best_cfg['val_best_acc_mean'], best_cfg['val_best_acc_ci'])}")
print(f"final_val_acc        : {fmt_mean_ci(best_cfg['final_val_acc_mean'], best_cfg['final_val_acc_ci'])}")
print(f"final_val_loss       : {fmt_mean_ci(best_cfg['final_val_loss_mean'], best_cfg['final_val_loss_ci'])}")
print(f"test_acc             : {fmt_mean_ci(best_cfg['test_acc_mean'], best_cfg['test_acc_ci'])}")
print(f"test_loss            : {fmt_mean_ci(best_cfg['test_loss_mean'], best_cfg['test_loss_ci'])}")
print(f"test_ce              : {fmt_mean_ci(best_cfg['test_ce_mean'], best_cfg['test_ce_ci'])}")
print(f"test_local           : {fmt_mean_ci(best_cfg['test_local_mean'], best_cfg['test_local_ci'])}")
print(f"avg_batch_loss_std   : {fmt_mean_ci(best_cfg['avg_batch_loss_std_mean'], best_cfg['avg_batch_loss_std_ci'])}")
print(f"avg_hidden_grad_std  : {fmt_mean_ci(best_cfg['avg_hidden_grad_std_mean'], best_cfg['avg_hidden_grad_std_ci'])}")

# ============================================================
# Compare best geometry-local against standard BP
# ============================================================
print("\n================ Best geometry-local vs Standard BP ================\n")
print(f"Standard BP   | test_acc={fmt_mean_ci(std_test_acc_mean, std_test_acc_ci)} | "
      f"test_loss={fmt_mean_ci(std_test_loss_mean, std_test_loss_ci)} | "
      f"best_val_acc={fmt_mean_ci(std_val_best_acc_mean, std_val_best_acc_ci)}")

print(f"Geometry-local| test_acc={fmt_mean_ci(best_cfg['test_acc_mean'], best_cfg['test_acc_ci'])} | "
      f"test_loss={fmt_mean_ci(best_cfg['test_loss_mean'], best_cfg['test_loss_ci'])} | "
      f"best_val_acc={fmt_mean_ci(best_cfg['val_best_acc_mean'], best_cfg['val_best_acc_ci'])}")

# ============================================================
# Optional: show all configurations sorted
# ============================================================
print("\n================ All geometry-local configurations (sorted) ================\n")
for r in geometry_results_sorted:
    print(
        f"lambda={r['lambda_local']:<4} | "
        f"protos={r['num_prototypes']:<2} | "
        f"tau={r['tau']:<3} | "
        f"best_val_acc={r['val_best_acc_mean']:.4f} | "
        f"test_acc={r['test_acc_mean']:.4f} | "
        f"test_loss={r['test_loss_mean']:.4f}"
    )

Using GPU: Tesla V100-PCIE-32GB

================ Running Standard BP baseline ================

Seed 42


100%|██████████| 26.4M/26.4M [00:01<00:00, 17.1MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 1.42MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 13.9MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 25.8MB/s]


  test_loss=0.4603 | test_acc=0.8414 | best_val_acc=0.8470 | epochs_to_target=nan
Seed 43
  test_loss=0.4624 | test_acc=0.8399 | best_val_acc=0.8610 | epochs_to_target=18
Seed 44
  test_loss=0.4507 | test_acc=0.8444 | best_val_acc=0.8520 | epochs_to_target=24
Seed 45
  test_loss=0.4611 | test_acc=0.8371 | best_val_acc=0.8570 | epochs_to_target=25
Seed 46
  test_loss=0.4643 | test_acc=0.8366 | best_val_acc=0.8540 | epochs_to_target=22

================ Running Geometry-local grid search ================

Total geometry-local configurations: 36

[01/36] lambda=0.0, num_prototypes=4, tau=0.5
  val_best_acc=0.8518 ± 0.0087 | test_acc=0.8367 ± 0.0017 | test_loss=0.4787 ± 0.0105

[02/36] lambda=0.0, num_prototypes=4, tau=1.0
  val_best_acc=0.8526 ± 0.0087 | test_acc=0.8378 ± 0.0012 | test_loss=0.4760 ± 0.0169

[03/36] lambda=0.0, num_prototypes=4, tau=3.0
  val_best_acc=0.8490 ± 0.0080 | test_acc=0.8370 ± 0.0015 | test_loss=0.4650 ± 0.0036

[04/36] lambda=0.0, num_prototypes=8, tau=0.5
  val